# LLM Gateway Basics - with LiteLLM

> Code + full series: **[github.com/dearnidhi/ai-security-bootcamp](https://github.com/dearnidhi/ai-security-bootcamp)**

**What is a gateway?**
A proxy layer between your app and the LLM provider (Groq, OpenAI, Anthropic...) that gives you retry, fallback, caching, load-balancing, and logging - without touching your business logic.

```
Your App  ->  Gateway (LiteLLM)  ->  Groq
```

**In this notebook we'll use LiteLLM** - an open-source Python library that unifies 100+ LLM providers behind one standard interface (`completion()`). No dashboard signup needed, everything runs in code.

We'll use only **Groq's free API** - for every gateway concept (fallback, load-balance, etc.) we'll use **two different Groq models**: `qwen/qwen3.8-27b` (primary, bigger) and `openai/gpt-oss-20b` (fallback, smaller/faster).

## 0. Setup

In [1]:
%pip install -q litellm python-dotenv groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Get a free key: https://console.groq.com/keys
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

PRIMARY_MODEL = "groq/qwen/qwen3.8-27b"
FALLBACK_MODEL = "groq/openai/gpt-oss-20b"

# gpt-oss models "think" before they answer. This keeps that thinking short,
# so the reply is never cut off. (Only the fallback model needs it.)
FALLBACK_PARAMS = {"reasoning_effort": "low"}

## 1. Baseline - Without a Gateway

A direct Groq SDK call. This works, but you get no retry/fallback/caching/observability - you'd have to write all of that yourself.

In [3]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
resp = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[{"role": "user", "content": "What is an LLM gateway in one line?"}],
)
print(resp.choices[0].message.content)

An LLM gateway is a centralized routing layer that abstracts away the specific details of different Large Language Model providers, allowing applications to seamlessly switch between or load-balance multiple LLMs via a single, unified interface.


This is a direct call - if Groq goes down or you hit a rate limit, your app crashes. **This is exactly the problem a gateway solves** - see below.

## 2. One Interface via LiteLLM

LiteLLM's `completion()` function takes an OpenAI-style call and uses the `model="groq/model-name"` prefix to decide where to send it. Switching models = just change the string.

In [4]:
from litellm import completion

response = completion(
    model=PRIMARY_MODEL,
    messages=[{"role": "user", "content": "What is an LLM gateway in one line?"}],
)
print("27b model:", response.choices[0].message.content)

response = completion(
    model=FALLBACK_MODEL,
    messages=[{"role": "user", "content": "What is an LLM gateway in one line?"}],
    **FALLBACK_PARAMS,
)
print("20b model:", response.choices[0].message.content)

27b model: An LLM gateway is a centralized proxy layer that provides unified access, routing, security, and observability for multiple large language model APIs under a single interface.


20b model: An LLM gateway is a unified interface that routes user requests to one or more large language model services, handling authentication, orchestration, and response formatting.


Same messages, same function call - only the `model=` string changed. This is the first benefit of a gateway: **model-agnostic code**.

## 3. Automatic Retries

On a rate limit (429) or a transient server error, LiteLLM retries automatically - before your app ever sees a failure.

In [5]:
response = completion(
    model=PRIMARY_MODEL,
    messages=[{"role": "user", "content": "Explain retries in one line."}],
    num_retries=3,
)
print(response.choices[0].message.content)

Retries are a mechanism to automatically re-execute a failed operation, hoping that a transient error (like a network glitch or server overload) has been resolved by the next attempt.


## 4. Request Timeouts

If a model stalls, set a hard time limit so your app doesn't block.

In [6]:
response = completion(
    model=PRIMARY_MODEL,
    messages=[{"role": "user", "content": "Explain timeouts in one line."}],
    timeout=10,  # seconds
)
print(response.choices[0].message.content)

A timeout is a predefined limit on the duration of an operation or communication, after which the system automatically stops waiting and assumes the process has failed or hung.


## 5. Fallbacks - Primary Fails, Backup Takes Over

Use `Router` to define a primary model plus a fallback. If the primary fails (4xx/5xx), the router automatically tries the fallback model - the caller never sees the failure.

In [7]:
from litellm import Router

router = Router(
    model_list=[
        {
            "model_name": "primary",
            "litellm_params": {"model": PRIMARY_MODEL},
        },
        {
            "model_name": "backup",
            "litellm_params": {"model": FALLBACK_MODEL, **FALLBACK_PARAMS},
        },
    ],
    fallbacks=[{"primary": ["backup"]}],
)

response = router.completion(
    model="primary",
    messages=[{"role": "user", "content": "Explain fallback routing in one line."}],
)
print(response.choices[0].message.content)

Fallback routing is a strategy that directs traffic to a secondary resource or backup path when a primary route fails to ensure service continuity.


## 6. Load Balancing - Split Traffic by Weight

Two entries with the same `model_name` pointing at two different Groq models - the LiteLLM Router automatically splits traffic between them (weighted).

In [8]:
lb_router = Router(
    model_list=[
        {
            "model_name": "chat-model",
            "litellm_params": {"model": PRIMARY_MODEL},
            "weight": 0.7,
        },
        {
            "model_name": "chat-model",
            "litellm_params": {"model": FALLBACK_MODEL, **FALLBACK_PARAMS},
            "weight": 0.3,
        },
    ]
)

for i in range(5):
    response = lb_router.completion(
        model="chat-model",
        messages=[{"role": "user", "content": f"Say hello, request #{i}"}],
    )
    print(i, "->", response.model, "|", response.choices[0].message.content[:60])

0 -> qwen/qwen3.8-27b | Hello! How can I help you today?


1 -> openai/gpt-oss-20b | Hello! How can I help you today?


2 -> openai/gpt-oss-20b | Hello! This is request #2.
3 -> qwen/qwen3.8-27b | Hello!


4 -> openai/gpt-oss-20b | Hello! 🎉  
(That’s the response for **request #4**.)


## 7. Response Caching

If the same prompt comes in again, don't call the model at all - return the cached response instantly.

In [9]:
import time
import litellm
from litellm.caching.caching import Cache

litellm.cache = Cache()  # in-memory cache

prompt = [{"role": "user", "content": "What is caching in 5 words?"}]

t0 = time.time()
r1 = completion(model=PRIMARY_MODEL, messages=prompt, caching=True)
print("1st call:", round(time.time() - t0, 2), "s ->", r1.choices[0].message.content)

t0 = time.time()
r2 = completion(model=PRIMARY_MODEL, messages=prompt, caching=True)
print("2nd call (cached):", round(time.time() - t0, 2), "s ->", r2.choices[0].message.content)

1st call: 0.35 s -> Storing data for quick retrieval.
2nd call (cached): 0.07 s -> Storing data for quick retrieval.


## 8. Observability - Cost & Token Tracking

Every response comes with `response.usage`, and `litellm.completion_cost()` gives you cost/usage - without any external dashboard.

In [10]:
response = completion(
    model=PRIMARY_MODEL,
    messages=[{"role": "user", "content": "One line on observability."}],
)
print("Response:", response.choices[0].message.content)
print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

try:
    cost = litellm.completion_cost(completion_response=response, model=PRIMARY_MODEL)
    print("Estimated cost ($):", cost)
except Exception:
    print("Estimated cost ($): not available for Groq free tier")

Response: Observability is the ability to understand the internal state of a system by analyzing its external outputs, such as logs, metrics, and traces.
Prompt tokens: 18
Completion tokens: 29
Total tokens: 47
Estimated cost ($): 0.0001304


## 9. Streaming

Streaming works through the gateway too - every chunk is passed through, and retries/fallback still apply.

In [11]:
stream = completion(
    model=PRIMARY_MODEL,
    messages=[{"role": "user", "content": "Explain LLM gateways in 3 bullet points."}],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

*

 **

Unified

 Interface

 &

...

 multi

-

tenant

 environments

.

## 10. Putting It All Together - Production-style Router

Fallback + retries + timeout + cooldown, all in one `Router` config - this is the same pattern used in the next project (the FastAPI app).

In [12]:
production_router = Router(
    model_list=[
        {
            "model_name": "chat",
            "litellm_params": {"model": PRIMARY_MODEL, "timeout": 10},
        },
        {
            "model_name": "chat",
            "litellm_params": {"model": FALLBACK_MODEL, "timeout": 10, **FALLBACK_PARAMS},
        },
    ],
    fallbacks=[{"chat": ["chat"]}],
    num_retries=2,
    cooldown_time=5,
)

response = production_router.completion(
    model="chat",
    messages=[{"role": "user", "content": "Summarize this notebook in one line."}],
)
print(response.choices[0].message.content)

A concise one‑sentence overview of the notebook’s purpose and main results.


---
## Recap

| Concept | What it solves |
| --- | --- |
| `completion(model="groq/model")` | One interface, easy model switch |
| `num_retries` | Transient errors handled automatically |
| `timeout` | Stuck requests don't block your app |
| `Router` + `fallbacks` | Primary down -> backup takes over silently |
| `Router` weighted `model_list` | Traffic split across models |
| `litellm.cache` | Repeat prompts cost $0 |
| `litellm.completion_cost()` | Cost tracking without a dashboard |
| `stream=True` | Real-time output, gateway features still apply |

**Next:** We'll use all of these concepts in a real FastAPI project - see the `project/` folder.